In [32]:
#start off running this
# import keras so it is defined for later use
import keras
import keras_nlp

# Load the saved model in .keras format
gemma_lm = keras.models.load_model('my_model.keras')

I0000 00:00:1731881873.182897   56813 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13764 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
/opt/conda/lib/python3.10/site-packages/keras/src/saving/serialization_lib.py:734: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
2024-11-17 22:19:25.601272: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2097152000 exceeds 10% of free system memory.
/opt/conda/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved o

In [ ]:
# Install Gradio
!pip install gradio


In [31]:
import gradio as gr
import tensorflow as tf
import keras
import speech_recognition as sr
from google.cloud import texttospeech

# Initialize the Text-to-Speech client
client = texttospeech.TextToSpeechClient()

# Initialize the Speech Recognition
recognizer = sr.Recognizer()

# Function to process Speech-to-Text
def speech_to_text(audio):
    try:
        with sr.AudioFile(audio) as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data)
            return text
    except sr.UnknownValueError:
        return "Sorry, I couldn't understand the audio. Please try again."
    except sr.RequestError as e:
        return f"Request error from Google STT service; {e}"

# Function to generate response from the chatbot model
def generate_response(model: keras.Model, question: str, max_length: int = 250) -> str:
    prompt_template = """ Role: You are a helpful real estate assistant providing support to legal matters in Ontario, 
    Canada Region; always answer the questions based on the following instructions; given the question, 
    generate an answer in 5 to 6 sentences always. For every statement in the answer, provide supporting 
    legal document or law segments from Ontario, Canada region.
    Question:\n{question}\n\nAnswer:\n"""
    
    prompt = prompt_template.format(question=question)
    
    try:
        generated_response = model.generate(prompt, max_length=max_length)
        generated_text = generated_response.decode('utf-8') if isinstance(generated_response, bytes) else generated_response
        answer_text = generated_text[len(prompt):].strip()
        return answer_text
    except Exception as e:
        return f"⚠️ Error generating answer: {e}"

# Function to synthesize text using Google Text-to-Speech
def synthesize_text(text):
    input_text = texttospeech.SynthesisInput(text=text)
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name="en-US-Standard-C",
        ssml_gender=texttospeech.SsmlVoiceGender.FEMALE
    )
    audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)
    response = client.synthesize_speech(
        request={"input": input_text, "voice": voice, "audio_config": audio_config}
    )
    audio_file_path = "output.mp3"
    with open(audio_file_path, "wb") as out:
        out.write(response.audio_content)
    return audio_file_path

# Combined function to handle chatbot response and TTS
def chatbot_with_tts_and_stt(audio):
    # Convert speech to text
    user_input = speech_to_text(audio)
    if not user_input.strip():
        return "Chatbot: I'm here to help! Please try speaking again.", None
    
    # Confirm the input text
    confirmation_message = f"Recognized text: '{user_input}'. Is this correct? If so, proceed. If not, try again."
    # Optionally, you can add a confirmation mechanism here
    
    # Generate chatbot response
    answer = generate_response(gemma_lm, user_input, max_length=5000)
    
    # Synthesize the generated text into speech
    audio_file_path = synthesize_text(answer)
    
    return answer, audio_file_path

# Create Gradio interface
interface = gr.Interface(
    fn=chatbot_with_tts_and_stt,
    inputs=gr.Audio(sources=["microphone"], type="filepath"),
    outputs=[gr.Textbox(), gr.Audio(autoplay=True)],
    title="Chatbot with STT and TTS",
    description="Speak your question and get a spoken answer."
)

# Launch the interface
interface.launch(share=True)



2024-11-17 22:15:28.492718: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731881728.515999   56813 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731881728.522972   56813 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-17 22:15:28.550151: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


* Running on local URL:  http://127.0.0.1:7880
* Running on public URL: https://bdbe298d4baa6a7d23.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/gradio/queueing.py", line 624, in process_events
    response = await route_utils.call_process_api(
  File "/opt/conda/lib/python3.10/site-packages/gradio/route_utils.py", line 323, in call_process_api
    output = await app.get_blocks().process_api(
  File "/opt/conda/lib/python3.10/site-packages/gradio/blocks.py", line 2015, in process_api
    result = await self.call_function(
  File "/opt/conda/lib/python3.10/site-packages/gradio/blocks.py", line 1562, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
  File "/opt/conda/lib/python3.10/site-packages/anyio/to_thread.py", line 33, in run_sync
    return await get_asynclib().run_sync_in_worker_thread(
  File "/opt/conda/lib/python3.10/site-packages/anyio/_backends/_asyncio.py", line 877, in run_sync_in_worker_thread
    return await future
  File "/opt/conda/lib/python3.10/site-packages/anyio/_backends/_asyncio.py", line 8

Attempting transcription...
Transcription: how long do I have 


Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/gradio/queueing.py", line 624, in process_events
    response = await route_utils.call_process_api(
  File "/opt/conda/lib/python3.10/site-packages/gradio/route_utils.py", line 323, in call_process_api
    output = await app.get_blocks().process_api(
  File "/opt/conda/lib/python3.10/site-packages/gradio/blocks.py", line 2015, in process_api
    result = await self.call_function(
  File "/opt/conda/lib/python3.10/site-packages/gradio/blocks.py", line 1562, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
  File "/opt/conda/lib/python3.10/site-packages/anyio/to_thread.py", line 33, in run_sync
    return await get_asynclib().run_sync_in_worker_thread(
  File "/opt/conda/lib/python3.10/site-packages/anyio/_backends/_asyncio.py", line 877, in run_sync_in_worker_thread
    return await future
  File "/opt/conda/lib/python3.10/site-packages/anyio/_backends/_asyncio.py", line 8